# Cell typing – updated to match IHOPE project

WINDOWS VERSION

# Prep and preprocessing

In [ ]:
import pandas as pd
from anndata import AnnData, read_h5ad
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
#sklearn

Make sure you're in the parent directory:

In [ ]:
import sys
from pathlib import Path

#Specific
PROJECT_ROOT = Path("D:/Emmy/IHOPE_SpatialProteomics")

Load, filter and normalize your raw data in csv format. Also, make sure to specify which file you're working with to facilitate later analysis. All subsequent naming of files is based on this.

In [ ]:
from scripts import preprocessing
import importlib

# Reload to ensure notebook uses the latest version of the script
importlib.reload(preprocessing)

# Input path, here you need to write the entire raw data file name
csv_in = PROJECT_ROOT / "data" / "raw" / "IHOPE14_MedLN_TopRight.csv"

# Keep track of which sample you're working on:
basename = "IHOPE14_MedLN_TopRight"

# Output path, you can use this default format based on your basename:
csv_out = PROJECT_ROOT / "data" / "processed" / f"{basename}_cleaned_raw.csv"

# Clean raw CSV
preprocessing.clean_cell_columns(csv_in, csv_out)

If you don't need to preprocess any data, define base name anyway:

In [ ]:
basename = "IHOPE14_MedLN_BottomLeft"

In [ ]:
from scripts import preprocessing
import importlib

# Reload to ensure the latest version of the script is used
importlib.reload(preprocessing)

# Run preprocessing
input_file = PROJECT_ROOT / "data" / "processed" / f"{basename}_cleaned_raw.csv"
output_file = PROJECT_ROOT / "data" / "processed" / f"{basename}_cleaned_filtered.csv"

preprocessing.preprocess(input_file, output_file, plot=False)


**Normalize**

Choose between: "arcsinh", "z-score" and "none".

For arcsinh, you can specify a cofactor.

In [ ]:
from scripts.transforms import apply_transform

method = "arcsinh"
cofactor = 5.0    # Comment out if not applying arcsinh
input_file = PROJECT_ROOT / "data" / "processed" / f"{basename}_cleaned_filtered.csv"
output_file = PROJECT_ROOT / "data" / "processed" / f"{basename}_cleaned_filtered_arcsinh_cf5.0.csv"

df_out, markers, metadata, fig = apply_transform(
    input_file=input_file,
    method=method,
    cofactor=cofactor, # Comment out if not applying arcsinh
    output_file=output_file,
    save_plot=True
)

# Building Anndata and annotating marker positivity


If you want to start from a cleaned and normalized file:

In [ ]:
basename = "IHOPE20_Spleen"

In [ ]:
from scripts.anndata_helpers import load_and_build_anndata, save_h5ad

# Read file and create AnnData object
filename = PROJECT_ROOT / "data" / "processed" / f"{basename}_cleaned_filtered_arcsinh_cf5.0.csv"
adata = load_and_build_anndata(filename)

In [ ]:
print("adata summary: ")
print(adata)
print("adata X shape: ")
print(adata.X.shape)
print("adata spatial data dimensions: ")
print(adata.obsm['spatial'].shape)
print("adata var: ")
print(adata.var)
print("adata var names: ")
print(adata.var_names)

Apply marker positivity thresholds (GMM intersections):

In [ ]:
from scripts.annotation import compute_positivity_matrix

# Annotate all cells with GMM-based positivity for canonical markers, fallback to percentile if unimodal
adata, thresholds, best_gmms = compute_positivity_matrix(
    adata,
    quantile=0.8,
    random_state=0
)

In [ ]:
# Visualize the distributions and thresholds:
from scripts.annotation import plot_marker_gmm_adata
plot_marker_gmm_adata(adata,
                      thresholds,
                      best_gmms,
                      title_prefix=f"{basename} Marker: ",
                      save=False,
                      filename="GMM_histograms")

In [ ]:
print(f"{basename}_GMM_marker_thresholds.png")

# Optional: add high/low levels to marker of interest.

Specify marker name, quantiles to label as low and high, and a recommended plot for visualization.

In [ ]:
from scripts import annotation

annotation.add_intensity_tiers(
    adata,
    marker="CD38",
    low_q=0.33,
    high_q=0.9,
    plot=True
)

In [ ]:
from scripts import annotation

annotation.add_intensity_tiers(
    adata,
    marker="CD21",
    low_q=0.33,
    high_q=0.9,
    plot=True
)

# Save anndata object with progress so far

In [ ]:
# Save AnnData as h5ad
h5adpath = PROJECT_ROOT / "data" / "processed" / "anndata" / f"{basename}_filtered_arcsinh_cf5_GMM.h5ad"
save_h5ad(adata, h5adpath)

To load the same anndata:

In [ ]:
import anndata as ad

h5adpath = PROJECT_ROOT / "data" / "processed" / "anndata" / f"{basename}_filtered_arcsinh_cf5_GMM.h5ad"

adata = ad.read_h5ad(h5adpath)

# Rule-based cell typing

Each cell type will correspond to a column in the AnnData object, with a boolean True/False for every cell type in every cell.

In [ ]:
import scripts
import scripts.celltype_rules_IHOPE
import importlib

importlib.reload(scripts.celltype_rules_IHOPE)

In [ ]:
#If you need to load anndata:
import anndata as ad

basename = "IHOPE20_Spleen"
adata = ad.read_h5ad(
    f"../data/processed/anndata/{basename}_filtered_arcsinh_cf5_GMM.h5ad"
)

Cell typing:

In [ ]:
from scripts.celltype_rules_IHOPE import assign_cell_types_bool_IHOPE

adata = assign_cell_types_bool_IHOPE(adata)


Save anndata with cell type information:

In [ ]:
from scripts.anndata_helpers import save_h5ad

h5adpath = PROJECT_ROOT / "data" / "processed" / "anndata" / f"{basename}_filtered_arcsinh_cf5_GMM_IHOPE_celltypes.h5ad"
save_h5ad(adata, h5adpath)

# Visual diagnostics

Some FACS-style plots to "diagnose" how the thresholds look:

In [ ]:
from scripts.marker_plots import plot_marker_axes

plot_marker_axes(
    adata,
    x_marker="CD21",
    y_marker="CD38",
    base_mask="type_B",
    title="B cells: CD21 vs CD38 (intensity thresholds)"
)

In [ ]:
plot_marker_axes(
    adata,
    x_marker="CCR7",
    y_marker="CD45RA",
    base_mask="type_T",
    title="T cells: CCR7 vs CD45RA"
)

Heatmap of markers in cell types:

In [ ]:
from scripts.plotting import clustered_marker_heatmap

# all cell type columns
celltype_cols = [
    c for c in adata.obs.columns
    if c.startswith(("type_", "state_", "subtype_"))
]

matrix = clustered_marker_heatmap(
    adata,
    celltype_cols=celltype_cols,
    min_cells=15,
    base_fig_width=10,
    fig_height=12,
    cmap="viridis",
)


# Spatial visualization

In [ ]:
from importlib import reload
from scripts import spatial_plot
reload(scripts.spatial_plot)

In [ ]:
from scripts.spatial_plot import spatial_celltype_plot

celltype_cols = [c for c in adata.obs.columns if c.startswith("subtype_") and not c.endswith ("unassigned")]

#or use this for all:
#[c for c in adata.obs.columns if c.startswith(("type_", "state_", "subtype_"))]

# Spatial plot
spatial_celltype_plot(adata, celltype_cols, min_cells=25)


In [ ]:
print(f"{basename}_subtypes_spatial.png")

In [ ]:
from scripts.spatial_plot import spatial_celltype_plot

# T cell subtype columns
t_subtypes = [c for c in adata.obs.columns if c.startswith("subtype_T")]

# Plot
spatial_celltype_plot(
    adata,
    celltype_cols=t_subtypes,
    min_cells=15,
    size=10,
    alpha=0.7
)


In [ ]:
print(f"{basename}_subtypes_T_spatial.png")

In [ ]:
from scripts.spatial_plot import spatial_celltype_plot

# B cell subtype columns
b_subtypes = [c for c in adata.obs.columns if c.startswith("subtype_B_")]

# Plot
spatial_celltype_plot(
    adata,
    celltype_cols=b_subtypes,
    min_cells=15,
    size=10,
    alpha=0.7
)


In [ ]:
print(f"{basename}_subtypes_B_spatial.png")


In [ ]:
from scripts.spatial_plot import spatial_celltype_plot

import math
import matplotlib.pyplot as plt

celltypes = (
    [c for c in adata.obs.columns if c.startswith("type_")] +
    [c for c in adata.obs.columns if c.startswith("intermediate_")] +
    [c for c in adata.obs.columns if c.startswith("subtype_")]
)

# Filter by min_cells
celltypes = [c for c in celltypes if adata.obs[c].sum() >= 15]

n = len(celltypes)
cols = 4
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*4))

axes = axes.flatten()

for ax, ct in zip(axes, celltypes):

    mask = adata.obs[ct].astype(bool)

    ax.scatter(
        adata.obs.loc[mask, "x"],
        adata.obs.loc[mask, "y"],
        s=8,
        alpha=0.7
    )

    ax.set_title(ct, fontsize=10)
    ax.invert_yaxis()
    ax.axis("off")

# remove empty panels
for ax in axes[n:]:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
print(f"{basename}_spatial_grid.png")


# Cell type summary in sample

Use after loading data and running annotation!

This will be used later for statistical comparison between tissues.

In [ ]:
from scripts import summary_celltypes_IHOPE
importlib.reload(scripts.summary_celltypes_IHOPE)

In [ ]:
from scripts.summary_celltypes_IHOPE import summarize_celltypes_IHOPE

df_summary = summarize_celltypes_IHOPE(
    adata,
    filename=f"{basename}_filtered_arcsinh_cf5.0_IHOPE_summary.csv"
)


# Comparison

Time to compare between samples!

In [ ]:
import importlib
import scripts.comparison as comparison

importlib.reload(comparison)

from scripts.comparison import (
    load_celltype_summaries,
    pivot_for_heatmap,
    plot_celltype_heatmap,
    print_numeric_summary,
)


In [ ]:
df = load_celltype_summaries(
    summaries_dir=summaries_dir,
    basenames=basenames,
)

print_numeric_summary(df)

matrix = pivot_for_heatmap(df)

plot_celltype_heatmap(
    matrix,
    title="Cell type composition across samples (all levels)",
)


# BANKSY
For this part you need a saved .h5ad file containing an anndata object, and Python 12 is required (not a later version of Python). Note that BANKSY only identifies spatial domains and not cell types. This section provides the workflow from AnnData object to BANKSY domains, which can be used as a foundation for further analysis.

In [ ]:
import sys
from pathlib import Path

#Specific
PROJECT_ROOT = Path("")

In [ ]:
basename = "IHOPE14_MedLN_BottomLeft"

In [ ]:
print(basename)

In [ ]:
import banksy
import scanpy as sc

from banksy.initialize_banksy import initialize_banksy
from banksy.run_banksy import run_banksy_multiparam


h5ad_file = PROJECT_ROOT / "data" / "processed" / "anndata" / f"{basename}_filtered_arcsinh_cf5_GMM_IHOPE_celltypes.h5ad"

# Load your spatial transcriptomics data
adata = sc.read_h5ad(h5ad_file)

# Initialize BANKSY
coord_keys = ('x', 'y', 'spatial')
banksy_dict = initialize_banksy(
    adata,
    coord_keys=coord_keys,
    num_neighbours=15,
    nbr_weight_decay='scaled_gaussian'
)

# Run BANKSY clustering

results_df = run_banksy_multiparam(
    adata,
    banksy_dict,
    lambda_list=[0.2],
    resolutions=[0.5, 1.0]
)

In [ ]:
from banksy.main import median_dist_to_nearest_neighbour
from banksy.initialize_banksy import initialize_banksy
from banksy.embed_banksy import generate_banksy_matrix
from banksy_utils.umap_pca import pca_umap
from banksy.cluster_methods import run_Leiden_partition
from banksy.plot_banksy import plot_results

# Parameters for BANKSY
coord_keys = ('x', 'y', 'spatial')
k_geom = 15
max_m = 1
nbr_weight_decay = "scaled_gaussian"
lambda_list = [0.8]
resolutions = [0.5]       # Leiden clustering resolution
pca_dims = [20]             # Try a lower number?
cluster_algorithm = 'leiden'
cmap = 'tab20'             # color map for spatial plotting
save_path = None           # e.g., "./BANKSY_results" if you want to save figures

# Compute neighbor distances
nbrs = median_dist_to_nearest_neighbour(adata, key=coord_keys[2])

# Initialize BANKSY
banksy_dict = initialize_banksy(
    adata,
    coord_keys,
    k_geom,
    nbr_weight_decay=nbr_weight_decay,
    max_m=max_m,
    plt_edge_hist=False,
    plt_nbr_weights=True,
    plt_agf_angles=False,
    plt_theta=False
)

# Generate BANKSY matrix
banksy_dict, banksy_matrix = generate_banksy_matrix(
    adata,
    banksy_dict,
    lambda_list,
    max_m
)

# Dimensionality reduction by UMAP
pca_umap(
    banksy_dict,
    pca_dims=pca_dims,
    add_umap=True
)

# Run Leiden clustering
results_df, max_num_labels = run_Leiden_partition(
    banksy_dict,
    resolutions=resolutions,
    num_nn=50,
    num_iterations=-1,
    partition_seed=1234,
    match_labels=True,
    max_labels=None
)

# Map clusters back to adata
cluster_labels = results_df.labels[results_df.index[0]].dense
adata.obs['banksy_domain'] = cluster_labels.astype(str)

# Optional: visualize
sc.pl.spatial(adata, color='banksy_domain', spot_size=30, title='BANKSY Domains')

# Optional: use BANKSY plotting function
if save_path is not None:
    os.makedirs(save_path, exist_ok=True)
    weights_graph = banksy_dict['scaled_gaussian']['weights'][1]
    plot_results(
        results_df[results_df['num_labels']==len(np.unique(cluster_labels))],
        weights_graph,
        cmap,
        match_labels=True,
        coord_keys=coord_keys,
        max_num_labels=max_num_labels,
        save_path=save_path,
        save_fig=True,
        save_fullfig=True,
        dataset_name='Sample',
        save_labels=True
    )

print(f"BANKSY complete for {basename}! Domains added to `adata.obs['banksy_domain']`")

Save anndata object with BANKSY domains

In [ ]:
h5adpath = f"../data/processed/anndata/{basename}_filtered_arcsinh_cf5_GMM_IHOPE_celltypes_banksy.h5ad"
save_h5ad(adata, h5adpath)

# B-cell follicle identification

Identify B-cell enriched domains and the cell types that require BANKSY information

In [ ]:
import importlib
import scripts.banksy_domains

# Reload the module after editing
importlib.reload(scripts.banksy_domains)

All BANKSY domains:

In [ ]:
from scripts.banksy_domains import plot_banksy_domains

plot_banksy_domains(adata)

In [ ]:
print(f"{basename}_BANKSY_domains.png")

In [ ]:
# Identify follicles (once per BANKSY run)
from scripts.banksy_domains import identify_bcell_follicles, plot_bcell_follicles

min_fraction = 0.5

adata, follicle_stats = identify_bcell_follicles(
    adata,
    banksy_domain_key="banksy_domain",
    min_fraction_bcells=min_fraction,
)

plot_bcell_follicles(adata, sample_name = basename)

In [ ]:
print(f"{basename}_BANKSY_Bcell_frac_{min_fraction}_domains.png")

In [ ]:
import importlib
import scripts.celltype_rules_IHOPE
importlib.reload(scripts.celltype_rules_IHOPE)


In [ ]:
# Add TfH-like cells
from scripts.celltype_rules_IHOPE import add_TfH_like_cells

adata = add_TfH_like_cells(
    adata,
    follicle_key="B_follicle",
    plot=True,
    size=0.3,
    sample_name = basename
)


Named as:

In [ ]:
print(f"{basename}_TfH_spatial_minfrac_Bc_{min_fraction}.png")

Finally, save the updated anndata object with the TfH cells added.

In [ ]:
h5adpath = f"../data/processed/anndata/{basename}_filtered_arcsinh_cf5_GMM_IHOPE_celltypes_banksy_TfH.h5ad"
save_h5ad(adata, h5adpath)

New cell type summary

In [ ]:
from scripts.summary_celltypes_IHOPE import summarize_celltypes_IHOPE

df_summary = summarize_celltypes_IHOPE(
    adata,
    filename=f"{basename}_filtered_arcsinh_cf5.0_IHOPE_summary_withTfH.csv"
)
